# TP 1 — HOG + classifieur sur images

**Objectifs**

- Calculer et visualiser un descripteur HOG.
- Comparer pixels bruts vs HOG comme entrée d'un classifieur linéaire.
- Mesurer la différence sur deux jeux de difficulté croissante.

**Durée indicative : 55 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from skimage import exposure
from skimage.feature import hog
from sklearn.datasets import fetch_olivetti_faces, load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

## Exercice 1 — Visualiser un HOG

On charge le dataset `olivetti_faces` (400 images 64×64 de 40 personnes).

1. Prenez la première image et affichez-la.
2. Calculez `hog(image, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)`.
3. Affichez l'image HOG résultante (rehaussée avec `exposure.rescale_intensity`). Que voit-on ?

<details>
<summary>💡 Coup de pouce — visualiser un descripteur HOG</summary>

**🎯 But :** comprendre concrètement ce que HOG capture : des **petites flèches d'orientation des gradients** dans chaque cellule de l'image.

**Calculer HOG**

```python
from skimage.feature import hog
features, hog_image = hog(
    image,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,            # ← important pour récupérer l'image visuelle
)
```

Avec `visualize=True`, on récupère **deux choses** :
- `features` : vecteur 1D (le descripteur utile au classifieur).
- `hog_image` : image 2D visualisable avec des flèches d'orientation.

**Améliorer le contraste de la visualisation**

L'image HOG brute est très sombre (faibles intensités). Pour bien voir :

```python
from skimage import exposure
hog_disp = exposure.rescale_intensity(hog_image, in_range=(0, hog_image.max() * 0.5))
```

`rescale_intensity` étire les valeurs [0, max/2] sur [0, 1] → on voit nettement les flèches.

**Affichage côte à côte**

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image, cmap='gray');     axes[0].set_title('Image source')
axes[1].imshow(hog_disp, cmap='gray');  axes[1].set_title('HOG (flèches d\'orientation)')
for ax in axes: ax.axis('off')
```

**Ce que vous devriez observer**

Les flèches HOG **suivent les contours** : silhouette du visage, lignes du nez, des sourcils, du contour des yeux. C'est exactement l'info pertinente pour reconnaître un visage. Les zones uniformes (joues, fond) sont **sombres** (peu de gradient).

C'est la grande force de HOG : **représentation invariante à la couleur**, qui capture seulement la « structure » des contours.

</details>

In [2]:
faces = fetch_olivetti_faces(shuffle=True, random_state=0)
# faces.images shape (400, 64, 64), faces.target shape (400,)
print(faces.images.shape, faces.target.shape)

# TODO

(400, 64, 64) (400,)


## Exercice 2 — Pixels bruts vs HOG sur Olivetti

1. Préparez deux matrices de features :
   - `X_pixels` : chaque image aplatie en vecteur (shape `(400, 4096)`).
   - `X_hog`    : `hog(image)` pour chaque image — choisissez les paramètres comme dans l'exercice 1.
2. Faites un split train/test 70/30 stratifié par identité, `random_state=0`.
3. Entraînez un Pipeline `StandardScaler + LinearSVC(C=1.0, max_iter=5000)` sur chacun.
4. Affichez les accuracies sur le test et comparez.

<details>
<summary>💡 Coup de pouce — pixels bruts vs descripteurs HOG sur Olivetti</summary>

**🎯 But :** comparer les performances d'un SVM entraîné sur les **pixels bruts** vs sur des **descripteurs HOG**.

**Préparer X_pixels (vecteur de pixels aplatis)**

```python
from sklearn.datasets import fetch_olivetti_faces
faces = fetch_olivetti_faces()
y = faces.target                          # shape (400,) — 40 personnes × 10 images
X_pixels = faces.images.reshape(400, -1)  # shape (400, 4096) = 64×64
```

⚠️ **Le `-1`** : NumPy calcule automatiquement la 2e dim (4096 = 64×64).

**Préparer X_hog**

```python
from skimage.feature import hog
import numpy as np
X_hog = np.array([
    hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
    for img in faces.images
])
print(X_hog.shape)   # typiquement (400, 1764) — beaucoup moins que 4096
```

HOG **réduit** la dimensionnalité tout en capturant l'info pertinente. C'est précisément ce qu'on veut.

**Splitter avec stratification**

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_hog, y,
    test_size=0.25,
    stratify=y,                  # ← crucial sur Olivetti
    random_state=0,
)
```

⚠️ **`stratify=y` est essentiel** : 40 classes avec 10 exemples chacune. Sans stratification, certaines identités pourraient ne pas être dans train ou test → train/test mal équilibré, scores bruités.

**Entraîner et comparer**

```python
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

for name, X in [('pixels', X_pixels), ('hog', X_hog)]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='linear'))])
    pipe.fit(Xtr, ytr)
    print(f"{name}: {pipe.score(Xte, yte):.3f}")
```

Vous devriez observer un **gain HOG vs pixels** de plusieurs points. HOG capture la structure pertinente (contours) en jetant les pixels redondants.

</details>

In [3]:
# TODO


## Exercice 3 — Effet des hyperparamètres HOG

Faites varier `pixels_per_cell` parmi `[(4, 4), (8, 8), (16, 16)]`. Pour chaque taille de cellule :

- recalculez `X_hog`,
- ré-évaluez l'accuracy sur le test.

Tracez accuracy vs taille de cellule. Quelle taille marche le mieux pour des visages 64×64 ?

<details>
<summary>💡 Coup de pouce — effet des hyperparamètres HOG</summary>

**🎯 But :** voir concrètement le compromis **granularité / robustesse** dans le choix de `pixels_per_cell`.

**Boucle sur 3 tailles de cellule**

```python
for ppc in [(4, 4), (8, 8), (16, 16)]:
    X_hog = np.array([
        hog(img, orientations=9, pixels_per_cell=ppc, cells_per_block=(2, 2))
        for img in faces.images
    ])
    Xtr, Xte, ytr, yte = train_test_split(X_hog, y, test_size=0.25, stratify=y, random_state=0)
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='linear'))]).fit(Xtr, ytr)
    print(f"ppc={ppc} → dim={X_hog.shape[1]:5d} | score={pipe.score(Xte, yte):.3f}")
```

**Compromis à comprendre**

| `pixels_per_cell` | Dimensionnalité | Granularité | Sensibilité au bruit |
|---|---|---|---|
| (4, 4) | TRÈS grande | ultra-fine (yeux, sourcils, narines) | élevée |
| (8, 8) | Moyenne | fine (formes du visage) | équilibrée |
| (16, 16) | Faible | grosses (silhouette uniquement) | faible |

**Ce que vous devriez observer**
- **(4, 4)** : score élevé en train mais peut overfit si le dataset est petit (cas Olivetti).
- **(8, 8)** : sweet spot pour beaucoup de tâches « visage ». Compromis idéal.
- **(16, 16)** : trop grossier pour la reconnaissance d'identité (les détails fins sont perdus).

**Règle empirique**

Si la **structure à reconnaître** fait T pixels (par exemple un œil ≈ 16 px), choisir `pixels_per_cell` ≈ T/3 à T/4 pour capturer les détails à l'intérieur sans trop fragmenter.

</details>

In [4]:
# TODO
